# A-WAN WP-3 — MEASURED mJ/token via NVML (Colab, T4/L4 GPU)
Runtime -> Change runtime type -> **GPU**. Run all. Downloads `measured_energy.json`
at the end — drop it into the repo at `runs/measured_energy.json` and re-run
`python run_awan_wp3.py`: the energy figures switch from MODELED to MEASURED.

In [ ]:
!pip -q install "transformers==4.57.1" accelerate pillow num2words pynvml

In [ ]:
import time, threading, json, numpy as np, torch
from pynvml import nvmlInit, nvmlDeviceGetHandleByIndex, nvmlDeviceGetPowerUsage
nvmlInit(); H = nvmlDeviceGetHandleByIndex(0)

class PowerSampler:
    def __init__(self, hz=10): self.hz, self.samples, self.on = hz, [], False
    def _loop(self):
        while self.on:
            self.samples.append((time.time(), nvmlDeviceGetPowerUsage(H)/1000.0))
            time.sleep(1.0/self.hz)
    def start(self): self.on=True; self.samples=[]; self.t=threading.Thread(target=self._loop); self.t.start()
    def stop(self): self.on=False; self.t.join(); return np.array(self.samples)

s = PowerSampler(); s.start(); time.sleep(30); idle = s.stop()
P_idle = idle[:,1].mean(); print("idle %.1f W" % P_idle)

In [ ]:
from PIL import Image, ImageDraw
import numpy as np
CLASSES=("car","bus","truck","person","bicycle")
COLORS={"red":(200,40,40),"blue":(40,80,200),"green":(40,160,70),"yellow":(230,200,40),"white":(240,240,240),"black":(25,25,25)}
SIZE={"car":(46,22),"bus":(78,24),"truck":(66,26),"person":(10,10),"bicycle":(18,8)}
def make_scene(seed,w=640,h=420):
    rng=np.random.default_rng(seed); img=Image.new("RGB",(w,h),(88,92,96)); dr=ImageDraw.Draw(img)
    ry=h//2; dr.rectangle([0,ry-55,w,ry+55],fill=(60,60,64))
    for x in range(10,w,60): dr.rectangle([x,ry-2,x+28,ry+2],fill=(210,210,190))
    for oid in range(int(rng.integers(7,11))):
        cls=CLASSES[rng.integers(0,5)]; col=list(COLORS)[rng.integers(0,6)]; ow,oh=SIZE[cls]
        x=rng.uniform(ow,w-ow); y=rng.uniform(ry-48,ry+48-oh) if cls!="person" else rng.uniform(15,h-15-oh)
        (dr.ellipse if cls=="person" else dr.rectangle)([x,y,x+ow,y+oh],fill=COLORS[col])
    return img
scenes=[make_scene(s) for s in range(50)]

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
MID="HuggingFaceTB/SmolVLM2-500M-Video-Instruct"
proc=AutoProcessor.from_pretrained(MID)
model=AutoModelForImageTextToText.from_pretrained(MID, dtype=torch.bfloat16).to("cuda").eval()
PROMPT=("You see a top-down street scene. List every vehicle and person you see. "
        "Reply ONLY with JSON: {\"facts\":[{\"object\":\"car|bus|truck|person|bicycle\",\"attr\":\"<color>\",\"confidence\":0.0}]}")

In [ ]:
tok_in=tok_out=0; E_pre=E_dec=0.0; n_views=0
sampler=PowerSampler()
for img in scenes*5:
    msgs=[{"role":"user","content":[{"type":"image","image":img},{"type":"text","text":PROMPT}]}]
    inputs=proc.apply_chat_template(msgs,add_generation_prompt=True,tokenize=True,return_dict=True,return_tensors="pt").to("cuda")
    n_in=inputs["input_ids"].shape[1]
    sampler.start(); t0=time.time()
    with torch.no_grad(): _=model(**inputs)
    torch.cuda.synchronize(); t1=time.time()
    with torch.no_grad(): out=model.generate(**inputs,do_sample=False,max_new_tokens=160)
    torch.cuda.synchronize(); arr=sampler.stop(); t2=time.time()
    if len(arr)<3: continue
    P=np.interp(np.linspace(arr[0,0],arr[-1,0],200),arr[:,0],arr[:,1])-P_idle
    ts=np.linspace(arr[0,0],arr[-1,0],200)
    E_total=np.trapezoid(np.clip(P,0,None),ts)
    frac_pre=(t1-t0)/max(t2-t0,1e-9)
    E_pre+=E_total*frac_pre; E_dec+=E_total*(1-frac_pre)
    tok_in+=int(n_in); tok_out+=int(out.shape[1]-n_in); n_views+=1
print(n_views,"views",tok_in,"+",tok_out,"tokens, prefill %.1f J decode %.1f J"%(E_pre,E_dec))

In [ ]:
res={"prefill_mJ_per_tok":E_pre*1e3/max(tok_in,1),
     "decode_mJ_per_tok":E_dec*1e3/max(tok_out,1),
     "total_mJ":(E_pre+E_dec)*1e3,"tok_in":tok_in,"tok_out":tok_out,
     "n_views":n_views,"gpu":"colab-nvml","model":MID,"idle_W":float(P_idle)}
print(json.dumps(res,indent=1))
json.dump(res,open("measured_energy.json","w"))
from google.colab import files; files.download("measured_energy.json")